In [ ]:
# Assignment 9 — Vision-Based Cognitive System
# Landing AI / Cognitive Computing lab: two proposed systems
#
# This notebook contains practical starter implementations for:
# 1. Reading medicine-bottle labels from images using OCR.
# 2. Detecting facial expression/emotion labels using a pretrained model.
#
# These are classroom prototypes only. Medicine OCR must not be used as the sole source for dosing or treatment decisions; verify all label information with the package and a pharmacist/clinician.

In [ ]:
# Setup

In [ ]:
# Run in a notebook cell if packages are missing:
# %pip install easyocr opencv-python matplotlib transformers torch

In [ ]:
# System 1 — Medicine label reader

In [ ]:
# Use EasyOCR to read text in a bottle-label photo. The program extracts visible text and can search the extracted text for common label fields; it does not verify the medicine or prescribe usage.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import easyocr
from pathlib import Path

# Set this to the path of your own label image.
image_path = "medicine_label.jpg"
image = cv2.imread(image_path)
if image is None:
    print("Place a label image at medicine_label.jpg (or change image_path).")
else:
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8,6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.show()

    reader = easyocr.Reader(['en'], gpu=False)
    detected = reader.readtext(image)
    extracted_text = "\\n".join([item[1] for item in detected])
    print("OCR text:\\n", extracted_text)

In [ ]:
# Basic information extraction from OCR output (heuristic; verify carefully)
import re
def find_label_fields(text):
    patterns = {
        "expiry_date": r'(?:exp(?:iry)?|use by)[:\\s-]*([0-9]{1,2}[/-][0-9]{2,4}|[A-Za-z]{3,9}\\s*[0-9]{4})',
        "batch": r'(?:batch|lot)[:\\s-]*([A-Za-z0-9-]+)',
        "dosage_strength": r'\\b\\d+(?:\\.\\d+)?\\s*(?:mg|mcg|g|ml|%|IU)\\b'
    }
    result={}
    for key,pattern in patterns.items():
        matches=re.findall(pattern,text,re.I)
        result[key]=matches
    return result

# Example usage after OCR:
# print(find_label_fields(extracted_text))

In [ ]:
# Optional OCR preprocessing

In [ ]:
def preprocess_label(image):
    gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    gray=cv2.GaussianBlur(gray,(3,3),0)
    enhanced=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
    return enhanced
# If OCR is poor, try:
# improved = preprocess_label(image)
# print(reader.readtext(improved))

In [ ]:
# System 2 — Emotion-aware assistant using facial-expression detection

In [ ]:
# This implementation uses a Hugging Face pretrained facial-expression classifier. It reports the model's predicted emotion category and confidence scores. The categories are model outputs—not reliable judgments about a person's internal state.

In [ ]:
# Install if needed: %pip install transformers torch pillow
from transformers import pipeline
from PIL import Image

emotion_model = pipeline(
    "image-classification",
    model="trpakov/vit-face-expression"
)

face_image_path = "face.jpg"  # replace with a facial image you have permission to use
try:
    face_img=Image.open(face_image_path).convert("RGB")
    predictions=emotion_model(face_img)
    print("Emotion predictions:")
    for item in predictions:
        print(item['label'], round(item['score'],3))
except FileNotFoundError:
    print("Place an image at face.jpg or change face_image_path.")

In [ ]:
# Simple response logic (demo only)

In [ ]:
def supportive_response(predicted_label):
    label=predicted_label.lower()
    if 'sad' in label:
        return "Would you like to take a break or talk about what's on your mind?"
    if 'angry' in label:
        return "Would a short pause or breathing exercise be helpful?"
    if 'happy' in label:
        return "Glad to see a positive expression—what would you like to work on?"
    if 'fear' in label or 'disgust' in label:
        return "Would you like to pause and choose what would help you feel comfortable?"
    return "How are you feeling right now? You can tell me in your own words."

# Example, using the highest-scoring predicted label:
# print(supportive_response(predictions[0]['label']))

In [ ]:
# Design summary
# Medicine label assistant: image input → image enhancement → OCR → text display/field extraction → user verifies label details.
#
# Emotion-aware assistant: image input → pretrained expression classifier → confidence output → optional gentle response. Ask the user how they feel rather than treating the classifier's output as fact.
#
# Limitations: OCR can misread small, curved, reflective or blurry text. Expression classifiers can be inaccurate, biased, or misinterpret context. These systems should be transparent, optional, privacy-conscious, and not used for consequential medical or psychological decisions.